In [1]:
import torch
from torchvision import transforms
from torch.utils.data import DataLoader
from torch.utils.data import random_split

# 已定义的模块
from datasetv2 import ImageSequenceDataset 


In [ ]:
# 图像预处理
import os
from pathlib import Path
from model.model import OzonePredictor
from tqdm import tqdm
import time


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

# 超参数
# 提供根目录
notebook_path = Path().resolve().parent
data_dir = notebook_path / "data"
json_path = data_dir / "dataset.json"
json_path = os.path.normpath(json_path)

batch_size = 16
num_epochs = 50
lr = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
# 数据加载
dataset = ImageSequenceDataset(json_path)
dataset_size = len(dataset)
val_size = int(dataset_size * 0.2)  # 20% 的数据用于验证
train_size = dataset_size - val_size

train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=batch_size,  shuffle=True,num_workers=16, pin_memory=True, persistent_workers=True,prefetch_factor=2)                 # 预取2个batch)
val_loader = DataLoader(val_dataset, batch_size=batch_size, num_workers=16, pin_memory=True, shuffle=False,persistent_workers=True, prefetch_factor=2  )

# 模型构建
model = OzonePredictor(cnn_out_dim=512, lstm_hidden_dim=512, output_dim=849)
model.to(device)

# 优化器与损失函数
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# 训练循环
best_val_loss = float('inf')
train_loss_history = []
val_loss_history = []

# 打印训练配置
print("\n" + "="*50)
print(f"Starting Training with Configuration:")
print(f"- Device: {device}")
print(f"- Batch Size: {batch_size}")
print(f"- Epochs: {num_epochs}")
print(f"- Learning Rate: {lr:.0e}")
print(f"- Training Samples: {train_size}")
print(f"- Validation Samples: {val_size}")
print("="*50 + "\n")
start_time = time.time()

for epoch in range(num_epochs):
    epoch_start_time = time.time()
    
    # 训练阶段
    model.train()
    train_loss = 0.0
    train_batches = 0
    
    # 使用tqdm进度条
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
    
    for batch in train_pbar:
        if batch is None:
            print("invalid batch")
            continue
            
        imgs = batch['images'].to(device, non_blocking=True)
        npy = batch['npy'].to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, npy)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        train_batches += 1
        train_pbar.set_postfix({'loss': f"{loss.item():.4f}", 
                            'GPU': f"{torch.cuda.memory_allocated()/1024**3:.1f}GB"})

    avg_train_loss = train_loss / train_batches if train_batches > 0 else 0
    train_loss_history.append(avg_train_loss)
    
    # 验证阶段
    model.eval()
    val_loss = 0.0
    val_batches = 0
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
    
    with torch.no_grad():
        for batch in val_pbar:
            if batch is None:

                continue
                
            imgs = batch['images'].to(device)
            npy = batch['npy'].to(device)
            
            outputs = model(imgs)
            loss = criterion(outputs, npy)
            val_loss += loss.item()
            val_batches += 1
            val_pbar.set_postfix({'loss': f"{loss.item():.4f}"})
    
    avg_val_loss = val_loss / val_batches if val_batches > 0 else 0
    val_loss_history.append(avg_val_loss)
    
    # 保存最佳模型
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.pth')
        best_epoch = epoch + 1
    
    # 计算epoch耗时
    epoch_time = time.time() - epoch_start_time
    
    # 打印epoch总结
    print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
    print(f"  Time: {epoch_time:.1f}s")
    print(f"  Train Loss: {avg_train_loss:.4f} {'↓' if avg_train_loss < (train_loss_history[-2] if len(train_loss_history)>1 else float('inf')) else '↑'}")
    print(f"  Val Loss:   {avg_val_loss:.4f} {'↓' if avg_val_loss < best_val_loss else '↑'}")
    print(f"  Best Val Loss: {best_val_loss:.4f} (Epoch {best_epoch})")
    print("-"*50)
    torch.cuda.empty_cache()

# 训练结束总结
total_time = time.time() - start_time
print("\n" + "="*50)
print("Training Complete!")
print(f"Total Training Time: {total_time:.1f}s ({total_time/60:.1f} minutes)")
print(f"Best Validation Loss: {best_val_loss:.4f} achieved at Epoch {best_epoch}")
print("="*50 + "\n")



cuda


d:\biC\.conda\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\biC\.conda\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)



Starting Training with Configuration:
- Device: cuda
- Batch Size: 16
- Epochs: 50
- Learning Rate: 1e-04
- Training Samples: 8040
- Validation Samples: 2010



KeyboardInterrupt: 

In [ ]:
# 绘制损失曲线（可选）
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 5))
plt.plot(train_loss_history, label='Train Loss')
plt.plot(val_loss_history, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid()
plt.show()